# Multivariate Regression - RegKit API

In this project we revisit the naïve solution to the multivariate regression problem, but this time we focus on reusing the `regkit` library that lives in [`src/`](../src/). Our optimized solution can be found in the companion notebook [02_optimized_solution.ipynb](./02_optimized_solution.ipynb).

## Setup
---

In [ ]:
# Use this to download and install packages in the notebook if needed
# !pip install -q numpy sympy matplotlib scikit-learn tqdm

# Install our library in editable mode
!python -m pip install -q -e .. && echo "Installed regkit successfully"

# Alternatively, if you want to install in regular mode, uncomment the following line:
# !python -m pip install -q .. && echo "Installed regkit successfully."

In [ ]:
# Needed libraries
import sys
from pathlib import Path

import numpy as np
import sympy as sp
import matplotlib as mpl
import itertools as it      # needed for combinations and permutations
import sklearn as skl

# Needed submodules
from matplotlib import pyplot as plt    # needed for plotting
from sympy.polys.monomials import itermonomials     # needed for generating monomials
from tqdm import tqdm    # needed for progress bars
from sklearn.decomposition import PCA   # needed for PCA
from sklearn.manifold import TSNE       # needed for t-SNE

# # Allow importing the project library when running from notebooks/
# PROJECT_ROOT = Path.cwd()
# if not (PROJECT_ROOT / 'src').exists():
#     PROJECT_ROOT = PROJECT_ROOT.parent
# SRC_PATH = PROJECT_ROOT / 'src'
# if str(SRC_PATH) not in sys.path:
#     sys.path.append(str(SRC_PATH))

from regkit import (
    create_target_dataset,
    gen_exploration_dataset,
    plot_dataset,
    regression_naive,
    regression_optimized
)

# Check the versions
print("🐍 Python version:", sys.version)
print("📦 NumPy version:", np.__version__)
print("🔢 SymPy version:", sp.__version__)
print("📊 Matplotlib version:", mpl.__version__)
print("📈 scikit-learn version:", skl.__version__)


In [ ]:
# Commonly used symbols
x, y, z = sp.symbols('x y z')

## Math Exploration

***  



Suppose we wish to approximate a given multivariate function $f:\mathbb{R}^n\to\mathbb{R}^m$ using a finite family of candidate functions called the “basis” functions $\Phi={\phi_0,\, \phi_1,\,\dots,\, \phi_k}$. A natural way to view this task is as a projection problem.

First, consider the collection

$$
V=\{\,f\mid f:\mathbb{R}^n\to\mathbb{R}^m\,\}.
$$

Because $V$ is closed under addition and scalar multiplication, it forms a vector space.

We can introduce the inner product

$$
\langle f,g\rangle 
= \int_{\Omega} f(\vec{x}) \cdot g(\vec{x})\,\mathrm{d}\vec{x},
$$

where “$\cdot$” is the standard $\langle \cdot \rangle_{\R^n} $ dot product in $\mathbb{R}^m$.

For the integral to be finite we must restrict the set of available functions to

$$
V_{\Omega}=\{\,f\in V\mid f_i\in L^2(\Omega)\text{ for }i=1,\dots,m\,\},
$$

meaning each component is square‑integrable over the bounded, measurable domain $\Omega\subset\mathbb{R}^n$. With this restriction $(V_{\Omega},\langle\cdot,\cdot\rangle)$ becomes a Hilbert space.

Since the dot product splits into components, the inner product expands as

$$
\langle f,g\rangle=
\int_{\Omega}\sum_{i=1}^m f_i(\vec{x})\,g_i(\vec{x})\,\mathrm{d}\vec{x}
=\sum_{i=1}^m \int_{\Omega} f_i(\vec{x})\,g_i(\vec{x})\,\mathrm{d}\vec{x}.
$$

Choosing a convenient rectangular domain $\Omega=[a_1,\, b_1]\times\cdots\times[a_n,\, b_n]$ gives a concrete formula:

$$
\langle \, f(\vec x) ,\, g(\vec x) \,\rangle
  = \sum_{i=1}^m \,
    \underbrace{\int_{a_1}^{b_1}\!\cdots\!\int_{a_n}^{b_n}}_{n \text{ times}} \;
    f_i(\underbrace{x_1,\dots,x_n}_{n \text{ times}} ) \,
    g_i(\underbrace{x_1,\dots,x_n}_{n \text{ times}}) \;
    \underbrace{\mathrm{d}x_n\cdots\mathrm{d}x_1}_{n \text{ times}}
$$


If instead of a continuous function, our target is a sample of points, we can replace the integral with a sum and hope it works 🙏.

$$\langle f, g\rangle
= \sum_{i=1}^m \sum_{\vec{p_i}}^m f_i(\vec{p_i})\,g_i(\vec{p_i})$$


By restructuring the commonly used orthogonal equation, we can create the system:
$$G A = X Y$$

where $G$ is the Gram matrix formed by

$$ G_{ij} = \langle \phi_i ,\, \phi_j \rangle $$

In explicit block form, this problem becomes:

$$
\underbrace{
  \begin{bmatrix}
    \langle \phi_0, \phi_0\rangle
    & \langle \phi_0, \phi_1\rangle
    & \cdots
    & \langle \phi_0, \phi_n\rangle \\[0.75em]
    \langle \phi_1, \phi_0\rangle
    & \langle \phi_1, \phi_1\rangle
    & \cdots
    & \langle \phi_1, \phi_n\rangle \\[0.75em]
    \vdots & \vdots & \ddots & \vdots \\[0.5em]
    \langle \phi_n, \phi_0\rangle
    & \langle \phi_n, \phi_1\rangle
    & \cdots
    & \langle \phi_n, \phi_n\rangle
  \end{bmatrix}
}_{G}
\;
\underbrace{
\begin{bmatrix}
  a_0 \\[0.5em] a_1 \\[0.5em] \vdots \\[0.5em] a_n
\end{bmatrix}
}_A

=

\underbrace{
\begin{bmatrix}
  \langle f, \phi_0\rangle \\[0.75em]
  \langle f, \phi_1\rangle \\[0.75em]
  \vdots \\[0.5em]
  \langle f, \phi_n\rangle
\end{bmatrix}.
}_{XY}

$$



A very natural way to build a basis is by using polynomials. To construct such a basis, we must have the terms

$$\phi_k = \hat{e}_d \prod x_i^{\alpha_i} $$

where $\hat{e}_d$ is a unit vector with value $1$ in the component of index $d$ and value $0$ elsewhere.

$\prod x_i^{\alpha_i} $ forms the cross terms of $x_i$ variables with sum of powers equal to $p$.

Thus, the polynomial can be formulated as

$$P_r(\vec x) = \sum_{j = 0}^{r} \sum_{d = 1}^{m} \sum_{i = 1}^{n} \hat{e}_d \prod_{\sum_i \alpha_i = r} x_i^{\alpha_i} $$

Note that we must either use tensor operations, creating a vector index $\vec k$, or we create a mapping
$$(d, \vec \alpha) \to k$$
to use with our regular matrix operations. Mathematically, they should be equivalent.

This basis has the advantage of being straightforward to integrate and manipulate, so we can compute the inner products for reasonably sized problems. However, it is not orthogonal, so the resulting Gram matrix can become ill-conditioned.

### Using `regkit.regression_naive`

The helper `regkit.regression_naive` bundles the symbolic basis generation, Gram matrix assembly, and least-squares solve into a single call. It returns the SymPy coefficient vector, the symbolic approximation, and a NumPy-callable function just like our inline implementation did.

In [ ]:
# Example usage
X = np.array([[1, 2], [3, 4]])
Y = np.array([[1, 2], [3, 4]])
degree = 1
coeffs, approx, f_approx = regression_naive(X, Y, degree)

print("Coefficients:")
display(coeffs)

print("Approximation:")
display(approx)

Y_pred = f_approx(X).T
print("Predicted Y values:")
print(Y_pred)

## Testing
---

We will test our regression function on a few synthetic datasets first.

This spiral, high-dimensional manifold is far more interesting. Rather than recreating the generator here, we will call `regkit.gen_exploration_dataset`, which exposes the exact helper we used when building the library.

### Generating exploratory data with `regkit.gen_exploration_dataset`

We can sample the same exploratory manifold used in the paper via `gen_exploration_dataset(num_points, n, m, seed=None, noise=0.0)`. It returns the `(X, Y)` pair ready for downstream experiments.

To actually visualize it, we need to project it into three dimensions. The library already ships with `regkit.plot_dataset`, which wraps the visualization logic in a single call.

### Visualizing datasets with `regkit.plot_dataset`

The `plot_dataset` helper accepts an `(n, N)` input matrix and an  `(m, N)` output matrix and automatically chooses an appropriate visualization strategy (surface plots for low-dimensional cases or PCA/t-SNE projections otherwise).

In [ ]:
# Example usage:
X, Y = gen_exploration_dataset(200, 3, 2, seed=42, noise=0.01)
print("X shape:", X.shape)
print("Y shape:", Y.shape)

# Plot the dataset and save the projection used
reducer = plot_dataset(X, Y)

display(reducer)

Again, our implementation is not very efficient, but we can confirm that it works as expected!

In [ ]:
coeffs, approx, f_approx = regression_naive(X.T, Y.T, degree=3) 

approx

In [ ]:
Y_pred = f_approx(X.T).squeeze()
Y_pred.shape

In [ ]:
# Plot the predicted values using the same projection as before
plot_dataset(X, Y_pred, reducer=reducer)

With lower resolution:

In [ ]:
# Degree 1
coeffs, approx, f_approx = regression_naive(X.T, Y.T, degree=1)
Y_pred = f_approx(X.T).squeeze()

print("Free projection:")
plot_dataset(X, Y_pred)

print("Using previous projection:")
plot_dataset(X, Y_pred, reducer=reducer)

Note the effect of keeping the same projection versus letting the plot function redo the projection. Since the poor approximation led to larger differences on the points, the PCA produced a different base and thus resulted in a different projection. On both projections, we can see that the curve is different, but keeping the projection constant leads to a more trustworthy comparison.

In [ ]:
# Degree 2
coeffs, approx, f_approx = regression_naive(X.T, Y.T, degree=2)
Y_pred = f_approx(X.T).squeeze()
plot_dataset(X, Y_pred)

With higher resolution, we get much closer to the original surface, although it takes significantly longer to run.

Also, it is good to note that a higher degree might cause our regression process to become unstable, 
though we did not observe that behaviour under the tested payload.

In [ ]:
# Degree 10
coeffs, approx, f_approx = regression_naive(X.T, Y.T, degree=10)
Y_pred = f_approx(X.T).squeeze()
plot_dataset(X, Y_pred, reducer=reducer)

Now, we can study the error:

In [ ]:
max_degree = 5
errors = []

for degree in tqdm(range(1, max_degree + 1), desc="Degrees"):
    # perform regression
    coeffs, approx, f_approx = regression_naive(X.T, Y.T, degree, quiet=True)
    Y_pred = f_approx(X.T).squeeze()   # (N, m)

    # compute RMSE
    err = np.sqrt(np.mean((Y_pred - Y)**2))
    errors.append(err)

# Plot
plt.figure()
plt.plot(range(1, max_degree + 1), errors, marker='o')
plt.xlabel("Polynomial Degree")
plt.ylabel("RMSE")
plt.title("RMSE vs Polynomial Degree")
plt.xticks(range(1, max_degree + 1))
plt.grid(True)
plt.show()

# Log of RSME
err_log = np.log(errors)
plt.figure()
plt.plot(range(1, max_degree + 1), err_log, marker='o')
plt.xlabel("Polynomial Degree")
plt.ylabel("Log(RMSE)")
plt.title("Log(RMSE) vs Polynomial Degree")
plt.xticks(range(1, max_degree + 1))
plt.grid(True)
plt.show()

For completeness, let's repeat the experiment using the optimized solver.

In [ ]:
max_degree = 5
errors = []

for degree in tqdm(range(1, max_degree + 1), desc="Degrees"):
    # perform regression
    coeffs, approx, f_approx = regression_optimized(X.T, Y.T, degree, quiet=True)
    Y_pred = f_approx(X.T).squeeze()   # (N, m)

    # compute RMSE
    err = np.sqrt(np.mean((Y_pred.T - Y)**2))
    errors.append(err)

# Plot
plt.figure()
plt.plot(range(1, max_degree + 1), errors, marker='o')
plt.xlabel("Polynomial Degree")
plt.ylabel("RMSE")
plt.title("RMSE vs Polynomial Degree")
plt.xticks(range(1, max_degree + 1))
plt.grid(True)
plt.show()

# Log of RSME
err_log = np.log(errors)
plt.figure()
plt.plot(range(1, max_degree + 1), err_log, marker='o')
plt.xlabel("Polynomial Degree")
plt.ylabel("Log(RMSE)")
plt.title("Log(RMSE) vs Polynomial Degree")
plt.xticks(range(1, max_degree + 1))
plt.grid(True)
plt.show()

We can use a higher-degree approximation with our optimized implementation, though it might not lead to better results. In fact, an overly high degree can reduce numerical stability.

In [ ]:
max_degree = 20
errors = []

for degree in tqdm(range(1, max_degree + 1), desc="Degrees"):
    # perform regression
    coeffs, approx, f_approx = regression_optimized(X.T, Y.T, degree, quiet=True)
    Y_pred = f_approx(X.T).squeeze()   # (N, m)

    # compute RMSE
    err = np.sqrt(np.mean((Y_pred.T - Y)**2))
    errors.append(err)

# Plot
plt.figure()
plt.plot(range(1, max_degree + 1), errors, marker='o')
plt.xlabel("Polynomial Degree")
plt.ylabel("RMSE")
plt.title("RMSE vs Polynomial Degree")
plt.xticks(range(1, max_degree + 1))
plt.grid(True)
plt.show()

# Log of RSME
err_log = np.log(errors)
plt.figure()
plt.plot(range(1, max_degree + 1), err_log, marker='o')
plt.xlabel("Polynomial Degree")
plt.ylabel("Log(RMSE)")
plt.title("Log(RMSE) vs Polynomial Degree")
plt.xticks(range(1, max_degree + 1))
plt.grid(True)
plt.show()

## Target Problem
---

Now, let's work with our target problem. The helper `regkit.create_target_dataset` reproduces the MATLAB prototype we started from, so we can instantiate it directly.

### Loading the target dataset with `regkit.create_target_dataset`

Call `create_target_dataset()` to retrieve the structured matrix pair `(X, Y)` used throughout the paper. The helper mirrors the original MATLAB script.

We can represent the terms we will be using like this:

In [ ]:
I = sp.eye(4)
e1, e2, e3, e4 = [sp.ImmutableMatrix(I[:, i]) for i in range(4)]
x1, x2, x3, x4 = sp.symbols('x1 x2 x3 x4')

terms = [
    e1, e2, e3, e4,
    x1 * e1, x2 * e2, x3 * e3, x4 * e4
]

display(sp.MatAdd(*terms, evaluate=False))


Now, let's instantiate the dataset and see how it looks.

In [ ]:
X, Y = create_target_dataset()
print("X shape:", X.shape)
print("Y shape:", Y.shape)

reducer2 = plot_dataset(X, Y)

Now, let's do the regression. Since the specification only asked for degree 1, this will be fast with `regkit.regression_naive`.

In [ ]:
coeffs, approx, f_approx = regression_naive(X.T, Y.T, degree=1)

print('\n' + '='*40)
print("Coefficients:")
display(coeffs)

print("Approximation:")
display(approx)

print("Plot")
Y_pred = f_approx(X.T).squeeze()
plot_dataset(X, Y_pred, reducer=reducer2)

Now, for fun, let's see what a higher-degree approximation can do without changing any of the library calls.

In [ ]:
# Degree 3
coeffs, approx, f_approx = regression_naive(X.T, Y.T, degree=3)
Y_pred = f_approx(X.T).squeeze()
plot_dataset(X, Y_pred, reducer=reducer2)

In [ ]:
# Degree 5
coeffs, approx, f_approx = regression_naive(X.T, Y.T, degree=5)
Y_pred = f_approx(X.T).squeeze()
plot_dataset(X, Y_pred, reducer=reducer2)

Let's also try our new implementation for completeness.

In [ ]:
# Degree 5
coeffs, approx, f_approx = regression_optimized(X.T, Y.T, degree=5)
Y_pred = f_approx(X.T).squeeze().T
plot_dataset(X, Y_pred, reducer=reducer2)

Now, let's study how the error evolves as we increase the approximation degree. We will use the optimized version because of time constraints.

In [ ]:
max_degree = 10
errors = []

for degree in tqdm(range(1, max_degree + 1), desc="Degrees"):
    # perform regression
    coeffs, approx, f_approx = regression_optimized(X.T, Y.T, degree, quiet=True)
    Y_pred = f_approx(X.T).squeeze().T   # (N, m)

    # compute RMSE
    err = np.sqrt(np.mean((Y_pred - Y)**2))
    errors.append(err)

# Plot
plt.figure()
plt.plot(range(1, max_degree + 1), errors, marker='o')
plt.xlabel("Polynomial Degree")
plt.ylabel("RMSE")
plt.title("RMSE vs Polynomial Degree")
plt.xticks(range(1, max_degree + 1))
plt.grid(True)
plt.show()

# Log of RSME
err_log = np.log(errors)
plt.figure()
plt.plot(range(1, max_degree + 1), err_log, marker='o')
plt.xlabel("Polynomial Degree")
plt.ylabel("Log(RMSE)")
plt.title("Log(RMSE) vs Polynomial Degree")
plt.xticks(range(1, max_degree + 1))
plt.grid(True)
plt.show()